In [ ]:
# Install required libraries (if not already installed)
!pip install transformers datasets scikit-learn

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
import os
import json
import random
import shutil
from sklearn.model_selection import train_test_split

# =========================
# PATHS
# =========================
INPUT_PATH = "/content/drive/MyDrive/biomedical_text_generation/data/final_datasets/title_text_gen_ready.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/biomedical_text_generation/data/tokenized"
UNSEEN_PATH = "/content/drive/MyDrive/biomedical_text_generation/data/unseen"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(UNSEEN_PATH, exist_ok=True)

# =========================
# SETTINGS
# =========================
RANDOM_SEED = 42
TEST_SIZE = 0.2

# Title is short, abstract is long
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 1024

# Models
MODELS = {
    #"biot5_text_gen": "QizhiPei/biot5-base",
    #"t5_base_text_gen": "t5-base",
    "bart_base_text_gen": "facebook/bart-base",
    "biobart_v2_text_gen": "GanjinZero/biobart-v2-base",
}

# =========================
# LOAD DATA
# =========================
data = []

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            item = json.loads(line)
        except json.JSONDecodeError as e:
            print(f"Skipping bad JSON on line {line_num}: {e}")
            continue

        input_text = item.get("input", "")
        target_text = item.get("target", "")

        if not isinstance(input_text, str) or not isinstance(target_text, str):
            continue

        input_text = input_text.strip()
        target_text = target_text.strip()

        if input_text == "" or target_text == "":
            continue

        data.append({
            "input": input_text,
            "target": target_text
        })

print(f"Loaded valid examples: {len(data)}")

if len(data) == 0:
    raise ValueError("No valid examples found in the input file.")

# =========================
# SPLIT DATA
# =========================
random.seed(RANDOM_SEED)
random.shuffle(data)

train_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED
)

print(f"Train size: {len(train_data)}")
print(f"Test size: {len(test_data)}")

# Save unseen test set
unseen_file = os.path.join(UNSEEN_PATH, "title_text_gen_unseen.jsonl")
with open(unseen_file, "w", encoding="utf-8") as f:
    for item in test_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved unseen test set to: {unseen_file}")

# =========================
# TOKENIZATION FUNCTION
# =========================
def tokenize_and_save(dataset_list, checkpoint, model_name, split):
    print(f"\nTokenizing for {model_name} ({split})")

    tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=False)
    ds = Dataset.from_list(dataset_list)

    def preprocess(example):
        src = example["input"]   # title
        tgt = example["target"]  # abstract

        # Prefix for T5-family
        if checkpoint in ["t5-base", "QizhiPei/biot5-base"]:
            src = "generate abstract: " + src

        model_inputs = tokenizer(
            src,
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
            padding="max_length"
        )

        labels = tokenizer(
            tgt,
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            padding="max_length"
        )

        # Ignore padding in loss
        label_ids = labels["input_ids"]
        label_ids = [
            token_id if token_id != tokenizer.pad_token_id else -100
            for token_id in label_ids
        ]

        model_inputs["labels"] = label_ids
        return model_inputs

    tokenized = ds.map(preprocess, remove_columns=ds.column_names)

    out_dir = os.path.join(OUTPUT_DIR, model_name, split)

    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)

    os.makedirs(out_dir, exist_ok=True)
    tokenized.save_to_disk(out_dir)

    print(f"Saved tokenized {split} set to: {out_dir}")
    print(tokenized[0])

# =========================
# RUN FOR ALL MODELS
# =========================
for model_name, checkpoint in MODELS.items():
    tokenize_and_save(train_data, checkpoint, model_name, "train")
    tokenize_and_save(test_data, checkpoint, model_name, "test")

print("\nDone.")